# 3. Preprocessing and Baseline Evaluation

This notebook continues the data audit from Notebook 02. It prepares the selected features, applies the chronological train-test split and evaluates a median baseline using MAE.

In [1]:
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

import sys 
from pathlib import Path

# Make shared project functions available
src_path = (Path.cwd().parent / "src").resolve()
sys.path.insert(0, str(src_path))

from data_preparation import (
    load_and_clean_data,
    chronological_train_test_split
)

## 1. Data Preparation

Load the cleaned dataset, select the input features, and create a chronologial train-test-split. 
Separate features (X) and target (Y) for subsequent preprocessing and modelling.

In [2]:
df = load_and_clean_data("../data/AirQualityUCI.csv")

In [3]:
print(df.shape)
print(df["NO2(GT)"].isna().sum())

(9357, 15)
1642


In [4]:
features = [
    "PT08.S1(CO)",
    "PT08.S2(NMHC)",
    "PT08.S3(NOx)",
    "PT08.S4(NO2)",
    "PT08.S5(O3)",
    "T",
    "RH",
    "AH",    
]

target = "NO2(GT)"

train_df, test_df = chronological_train_test_split(df)

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))


Training rows: 5901
Test rows: 1814


In [5]:
X_train = train_df[features].copy()
X_test = test_df[features].copy()

Y_train = train_df[target].copy()
Y_test = test_df[target].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Y_train:", Y_train.shape)
print("Y_test:", Y_test.shape)



X_train: (5901, 8)
X_test: (1814, 8)
Y_train: (5901,)
Y_test: (1814,)


## 2. Preprocessing assessment

**Missing feature values and preprocessing decisions**


In [6]:
missing_counts = train_df[features].isna().sum()
missing_percentages = train_df[features].isna().mean()*100

missing_summary = pd.DataFrame({
    "Missing count": missing_counts,
    "Missing (%)": missing_percentages.round(2)
})

display(missing_summary)

print(
    "Training rows with at least one missing feature:",
    train_df[features].isna().any(axis=1).sum()
)

,Missing count,Missing (%)
PT08.S1(CO),239,4.05
PT08.S2(NMHC),239,4.05
PT08.S3(NOx),239,4.05
PT08.S4(NO2),239,4.05
PT08.S5(O3),239,4.05
T,239,4.05
RH,239,4.05
AH,239,4.05


Training rows with at least one missing feature: 239


In [7]:
missing_per_row = X_train.isna().sum(axis=1)

print("Missing features per training row:")
print(missing_per_row.value_counts().sort_index())

print("Rows with all 8 features missing:")
print((missing_per_row == len(features)).sum())


Missing features per training row:
0    5662
8     239
Name: count, dtype: int64
Rows with all 8 features missing:
239


In [8]:
fully_missing = X_train.isna().all(axis=1)

print("First affected timestamps:")
display(train_df.loc[fully_missing, ["Date", "Time"]].head(10))

print("Last affected timestamps:")
display(train_df.loc[fully_missing, ["Date", "Time"]].tail(10))

First affected timestamps:


,Date,Time
524,01/04/2004,14.00.00
525,01/04/2004,15.00.00
526,01/04/2004,16.00.00
701,08/04/2004,23.00.00
702,09/04/2004,00.00.00
703,09/04/2004,01.00.00
704,09/04/2004,02.00.00
706,09/04/2004,04.00.00
707,09/04/2004,05.00.00
708,09/04/2004,06.00.00


Last affected timestamps:


,Date,Time
7197,04/01/2005,15.00.00
7198,04/01/2005,16.00.00
7199,04/01/2005,17.00.00
7200,04/01/2005,18.00.00
7201,04/01/2005,19.00.00
7202,04/01/2005,20.00.00
7203,04/01/2005,21.00.00
7204,04/01/2005,22.00.00
7205,04/01/2005,23.00.00
7206,05/01/2005,00.00.00


In [9]:
groups = fully_missing.ne(fully_missing.shift()).cumsum()
missing_runs = fully_missing.groupby(groups).agg(["first", "size"])
missing_runs = missing_runs[missing_runs["first"]]

print("Number of missing-feature periods:", len(missing_runs))
print("Longest period (rows):", missing_runs["size"].max())
print("Median period length (rows)::", missing_runs["size"].median())


Number of missing-feature periods: 10
Longest period (rows): 72
Median period length (rows):: 14.0


**Missing feature values - findings**

Of the 5901 training observations, 239 (4,05%) have all eight input features missing.
No partially missing feature rows were found.

These observations occur in 10 consecutive groups of training rows, with a maximum length of 72 rows.

Since these rows contain no measured input features, they were excluded from feature-based training rather than imputed.
The remaining 5662 training observations contain no missing feature values

In [10]:
# Exclude training rows without any available input features
valid_feature_rows = X_train.notna().any(axis=1)

X_train_features = X_train.loc[valid_feature_rows].copy()
Y_train_features = Y_train.loc[valid_feature_rows].copy()

print("Original training rows:", len(X_train))
print("Rows retained for feature-based models:", len(X_train_features))
print("Rows excluded:", (~valid_feature_rows).sum())
print("Remaining missing feature values:", X_train_features.isna().sum().sum())


Original training rows: 5901
Rows retained for feature-based models: 5662
Rows excluded: 239
Remaining missing feature values: 0


In [11]:
# Apply the same feature-availability rule to the test set
valid_test_feature_rows = X_test.notna().any(axis=1)

X_test_features = X_test.loc[valid_test_feature_rows].copy()
Y_test_features = Y_test.loc[valid_test_feature_rows].copy()

print("Original test rows:", len(X_test))
print("Rows retained:", len(X_test_features))
print("Rows excluded:", (~valid_test_feature_rows).sum())

Original test rows: 1814
Rows retained: 1731
Rows excluded: 83


In [12]:
feature_preprocessing = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

feature_preprocessing

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness des

**Feature selection and leakage assesment**

Eight sensor and environmental features are selected. Other ground-truth pollutant measurements are excluded to reduce the risk of target leakage. 

The model estimates hourly NO2(GT) using measurements from the same hour, assuming these are available at prediction time. PT08.S4(NO2) is a sensor response, not the ground-truth target.

In [13]:
correlations = train_df[features + [target]].corr()

display(
    correlations[target]
    .drop(target)
    .sort_values(ascending=False)
)

PT08.S2(NMHC)    0.773376
PT08.S5(O3)      0.766353
PT08.S1(CO)      0.703066
PT08.S4(NO2)     0.383732
T               -0.039406
RH              -0.125097
AH              -0.232312
PT08.S3(NOx)    -0.638113
Name: NO2(GT), dtype: float64

Sensor features show stronger correlations with NO2(GT) than environmental variables.
All eight features are retained for inital modelling.

## 3. Baseline model and evaluation

No feature preprocessing is required for the DummyRegressor, as it only uses the training targets.

A median-based baseline was selected because MAE is the primary evaluation metric and the median minimises absolute errors.

In [14]:
baseline_model = DummyRegressor(strategy="median")
baseline_model.fit(X_train_features, Y_train_features)
baseline_model

,"strategy strategy: {""mean"", ""median"", ""quantile"", ""constant""}, default=""mean""Strategy to use to generate predictions.* ""mean"": always predicts the mean of the training set* ""median"": always predicts the median of the training set* ""quantile"": always predicts a specified quantile of the training set, provided with the quantile parameter.* ""constant"": always predicts a constant value that is provided by the user.",'median'
,"constant constant: int or float or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
,"quantile quantile: float in [0.0, 1.0], default=NoneThe quantile to predict using the ""quantile"" strategy. A quantile of0.5 corresponds to the median, while 0.0 to the minimum and 1.0 to themaximum.",None
Name,Type,Value
"constant_ constant_: ndarray of shape (1, n_outputs)Mean or median or quantile of the training targets or constant valuegiven by the user.","ndarray[float64](1, 1)",[[101.]]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](8,)","['PT08.S1(CO)','PT08.S2(NMHC)','PT08.S3(NOx)',...,'T','RH','AH']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,8
n_outputs_ n_outputs_: intNumber of outputs.,int,1


In [15]:
tscv = TimeSeriesSplit(n_splits=5)

baseline_mae_scores = -cross_val_score(
    DummyRegressor(strategy="median"),
    X_train_features,
    Y_train_features,
    cv=tscv,
    scoring="neg_mean_absolute_error"
)

print("MAE per fold:", baseline_mae_scores)
print("Mean MAE:", baseline_mae_scores.mean())
print("Standard deviation:", baseline_mae_scores.std())

MAE per fold: [28.46871686 33.05726405 35.91198303 41.06468717 37.97773065]
Mean MAE: 35.29607635206787
Standard deviation: 4.3005255420251


Five-fold TimeSeriesSplit was used to preserve temporal order during cross-validation. The median baseline is evaluated on training observations with available input features to ensure comparability with subsequent models.
MAE varies across folds, with higher errors in later periods, suggesting temporal variation in baseline performance. The held-out test set remains reserved for final evaluation.